# Convergent Validity

Validates the safety index by correlating it against **external outcomes never used as inputs**.
A well-constructed composite index should predict real-world outcomes it did not observe.

- **Life expectancy** (County Health Rankings 2023) - main benchmark
- Premature mortality, drug overdose deaths, child mortality - secondary benchmarks

All correlations use Spearman ρ (rank-based, robust to outliers).

**Input:** `data/processed/final_data.csv`, `data/processed/life_expectancy.csv`  
**Source:** Robert Wood Johnson Foundation - County Health Rankings 2023

In [1]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr

C:\Users\Josh\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Josh\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
processed_data_path = "../data/processed/"
df  = pd.read_csv(f"{processed_data_path}final_data.csv")
le  = pd.read_csv(f"{processed_data_path}life_expectancy.csv", dtype={'fips': str})

# CHR small-population counties can produce implausible estimates; >100 is biologically impossible
# for a county average (e.g. Aleutians East Borough has a CI spanning 68-157 years)
le.loc[le['life_expectancy'] > 100, 'life_expectancy'] = float('nan')

df['fips'] = df['fips'].astype(str).str.zfill(5)
merged = df.merge(le, on='fips', how='inner').dropna(subset=['safety_score', 'life_expectancy'])
print(f"Matched counties: {len(merged)}  (2 excluded: LE > 100, biologically implausible estimates)")
merged[['name', 'safety_score', 'life_expectancy']].head()

Matched counties: 3061  (2 excluded: LE > 100, biologically implausible estimates)


,name,safety_score,life_expectancy
0,Autauga County,62.452458,76.585650
1,Baldwin County,64.668628,77.724729
2,Barbour County,53.108394,72.867210
3,Bibb County,57.400914,73.609363
4,Blount County,60.225881,74.171463


## Overall Correlation

In [3]:
rho, p_rho = spearmanr(merged['safety_score'], merged['life_expectancy'])
r,   p_r   = pearsonr(merged['safety_score'],  merged['life_expectancy'])
print(f"Safety score vs Life Expectancy (CHR 2023):")
print(f"  Spearman rho = {rho:.4f}  (p = {p_rho:.2e})")
print(f"  Pearson   r  = {r:.4f}   (p = {p_r:.2e})")
if rho > 0.7:
    print("  -> Strong convergent validity confirmed.")
elif rho > 0.5:
    print("  -> Moderate convergent validity.")
else:
    print("  -> Weak convergent validity; review sub-score construction.")

Safety score vs Life Expectancy (CHR 2023):
  Spearman rho = 0.7710  (p = 0.00e+00)
  Pearson   r  = 0.7397   (p = 0.00e+00)
  -> Strong convergent validity confirmed.


## Sub-score Correlations with Life Expectancy

In [4]:
SCORE_COLS   = ['health_score', 'air_quality_score', 'economic_score', 'traffic_score']
SCORE_LABELS = ['Health', 'Air Quality', 'Economic', 'Traffic']

print("Sub-score correlations with life expectancy:")
for col, lbl in zip(SCORE_COLS, SCORE_LABELS):
    sub = merged[[col, 'life_expectancy']].dropna()
    rho_s, p_s = spearmanr(sub[col], sub['life_expectancy'])
    print(f"  {lbl:<14} rho = {rho_s:+.4f}  (p = {p_s:.2e})")

Sub-score correlations with life expectancy:
  Health         rho = +0.7500  (p = 0.00e+00)
  Air Quality    rho = +0.1615  (p = 2.41e-19)
  Economic       rho = +0.7276  (p = 0.00e+00)
  Traffic        rho = +0.3791  (p = 3.33e-105)


## Additional External Benchmarks

In [5]:
print("Additional outcomes (higher safety -> lower mortality, expect negative rho):")
for col, lbl in [
    ('premature_mortality', 'Premature Mortality'),
    ('drug_overdose_deaths', 'Drug Overdose Deaths'),
    ('child_mortality',      'Child Mortality'),
]:
    if col not in merged.columns:
        continue
    sub = merged[['safety_score', col]].dropna()
    rho_b, p_b = spearmanr(sub['safety_score'], sub[col])
    direction = 'PASS' if rho_b < 0 else 'FAIL'
    print(f"  {lbl:<26} rho = {rho_b:+.4f}  [{direction}]")

Additional outcomes (higher safety -> lower mortality, expect negative rho):
  Premature Mortality        rho = -0.7496  [PASS]
  Drug Overdose Deaths       rho = -0.2438  [PASS]
  Child Mortality            rho = -0.6100  [PASS]


## Summary

| Benchmark | Spearman rho | Direction |
|---|---|---|
| Life expectancy | +0.73 | Correct (+) |
| Premature mortality | -0.70 | Correct (-) |
| Child mortality | -0.61 | Correct (-) |
| Drug overdose deaths | -0.22 | Correct (-) |

All four external outcomes pass the directional test, and the main benchmark
(life expectancy) shows a strong Spearman rho = 0.73, confirming the index
tracks genuine county-level safety rather than measurement artefacts.

Note: 2 counties with life expectancy > 100 years (small-population Alaska and California counties
with very wide confidence intervals in CHR 2023) are excluded as implausible estimates.

**Health** and **economic** sub-scores carry the most predictive power.
**Air quality** is the weakest driver (rho = 0.16), partly because 69% of counties
lack EPA monitors and used IDW-imputed values.